# 1 - PII detection: which detector do we anonymise with?

**Question.** Ticket free text carries names, mailboxes, phone numbers and the
occasional social-security number nobody decided to collect. Which detector
removes them, at what cost, and what does each one miss?

**Why first.** Every other notebook here reads ticket text. If this is not
solved, none of them may run on real data at all.

| Approach | What it is |
|---|---|
| `rules` | Regex, contextual triggers, authored gazetteer |
| `transformer` | Multilingual HF NER checkpoint (PER/LOC) |
| `presidio` | Presidio analyser: spaCy + validated patterns |
| `gliner` | Zero-shot NER, told our entity names at inference |

**All four arms are mandatory.** They used to sit behind optional extras and a
missing one was skipped; a benchmark that silently drops a competitor is not a
benchmark, so every dependency is now a hard requirement and any failure here
raises.

**Every detector runs alone** (`with_structural=False`). The `transformer`
backend normally composes with the rule engine, and that composition is the
better *deployment* - but a composed detector cannot be benchmarked: its score
is the union of two systems, so a gap in one is hidden by the other.

```bash
uv sync                 # torch, spaCy, presidio, gliner - one heavy install
uv run itsm data build
```

**First run downloads model weights** (~1 GB across the three model-based
arms, cached afterwards). The spaCy pipeline Presidio needs is fetched
automatically - no manual `spacy download` step.

### Runtime

`SAMPLE = 400` keeps the whole notebook to a few minutes on CPU. The model arms
dominate: they are batched, and their pipelines are cached process-wide, so
section 5 reuses the weights section 2 loaded instead of reading them again.

In [1]:
import json
import time

import pandas as pd

from itsm.config import get_settings
from itsm.pipelines.anonymise import load_raw_tickets
from itsm.privacy.anonymiser import Anonymiser
from itsm.privacy.evaluate import (
    entity_distribution,
    residual_pii_rate,
    score_detection,
    spans_from_payload,
)

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

settings = get_settings()
COLUMN = 'description'   # the long field: most PII per ticket
SAMPLE = 400             # ~3 min on CPU across four arms; raise for a final run
SEED = 42
BACKENDS = ['rules', 'transformer', 'presidio', 'gliner']

## 1. Sample and ground truth

The generator plants the identifiers itself and records the **character
offsets** of each one. That is what turns the numbers below into measurements.
Support per entity matters when reading section 4: recall of 0.5 on four
annotated values is noise, not a finding.

In [2]:
tickets = load_raw_tickets(settings)
if 'pii_spans' not in tickets.columns:
    raise RuntimeError('no pii_spans annotation - run `uv run itsm data build` first')

sample = tickets.sample(n=min(SAMPLE, len(tickets)), random_state=SEED).reset_index(drop=True)
texts = sample[COLUMN].fillna('').astype(str).tolist()
references = [
    spans_from_payload(json.loads(payload or '{}'), COLUMN) for payload in sample['pii_spans']
]

support = pd.Series(entity_distribution(references), name='annotated values')
print(len(sample), 'tickets,', int(support.sum()), 'annotated values in', COLUMN)
support.to_frame()

400 tickets, 373 annotated values in description


,annotated values
PERSON,168
PHONE,51
ADDRESS,28
NATIONAL_ID,21
DATE_OF_BIRTH,20
IP_ADDRESS,19
IBAN,19
CREDIT_CARD,17
EMPLOYEE_ID,17
EMAIL,13


## 2. Run all four

Warm-up first, so the timing is throughput and not a one-off model load.

In [3]:
runs = {}

for backend in BACKENDS:
    anonymiser = Anonymiser(
        backend=backend,
        model_name=settings.pii_model(backend),
        language=settings.presidio_language,
        with_structural=False,
    )
    # Warm-up: loads (and on first ever run, downloads) the weights, so the
    # measured time below is inference and not model loading.
    anonymiser.anonymise_text('Merci de contacter Jean Dupont au 06 12 34 56 78.')

    started = time.perf_counter()
    outcome = anonymiser.anonymise_column(texts)
    elapsed = time.perf_counter() - started

    runs[backend] = {
        'detection': score_detection(references, outcome.spans, texts=texts),
        'residual': residual_pii_rate(outcome.anonymised, references),
        'ms_per_ticket': 1000 * elapsed / len(texts),
    }
    print(f'{backend:12s} {outcome.n_spans:5d} spans  {elapsed:6.1f}s'
          f'  {1000 * elapsed / len(texts):7.1f} ms/ticket')

rules          373 spans     0.0s      0.0 ms/ticket


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

transformer    177 spans     1.3s      3.2 ms/ticket


/mnt/c/Users/AmineDIAB/projects/Thales/ITSM/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Using Python 3.12.14 environment at: /mnt/c/Users/AmineDIAB/projects/Thales/ITSM/.venv
Resolved 1 package in 29.59s
 Downloaded en-core-web-md
Prepared 1 package in 28.10s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 2.07s
 + en-core-web-md==3.8.0 (from https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.8.0/en_core_web_md-3.8.0-py3-none-any.whl)


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
presidio       286 spans     2.2s      5.6 ms/ticket


...

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spm.model: downloading bytes:           |  0.00B            

/mnt/c/Users/AmineDIAB/projects/Thales/ITSM/src/itsm/privacy/gliner.py:281: FutureWarning: GLiNER.batch_predict_entities is deprecated and will be removed in a future release. Please use GLiNER.inference instead.
  raw = batched(chunk, names, threshold=self.min_score)


gliner         435 spans    31.1s     77.8 ms/ticket


## 3. Headline

Rows are ordered by what decides. **`residual values in output`** is the only
one that cannot be gamed by generous span matching: the original values are
searched for in the rewritten text. Span metrics measure the detector; this
measures the deliverable.

`partial residuals` is the row people forget - `Jean Dupont` redacted from
`Dupont` onwards leaves `Jean` behind. Span arithmetic scores that as a hit; a
reader of the output still knows who it is.

In [4]:
def headline(run):
    detection, residual = run['detection'], run['residual']
    recall, precision = detection['recall_coverage'], detection['precision_coverage']
    f1 = 2 * recall * precision / (recall + precision) if recall + precision else 0.0
    return {
        'residual values in output': residual['residual_values'],
        'residual rate, direct ids': round(residual['residual_rate_direct_identifiers'], 4),
        'partial residuals': residual['residual_partial_values'],
        'coverage recall': round(recall, 4),
        'precision': round(precision, 4),
        'F1': round(f1, 4),
        'false-positive spans': detection['false_positive_spans'],
        'over-redacted chars %': round(100 * detection['over_redacted_char_share'], 3),
        'ms per ticket': round(run['ms_per_ticket'], 2),
    }


comparison = pd.DataFrame({backend: headline(run) for backend, run in runs.items()})
comparison

,rules,transformer,presidio,gliner
residual values in output,0.00,207.0000,164.0000,13.0000
"residual rate, direct ids",0.00,0.4256,0.3668,0.0277
partial residuals,0.00,0.0000,5.0000,0.0000
coverage recall,1.00,0.4450,0.5362,0.9651
precision,1.00,0.9379,0.6993,0.8276
F1,1.00,0.6036,0.6070,0.8911
false-positive spans,0.00,11.0000,86.0000,75.0000
over-redacted chars %,0.00,0.1300,4.7000,1.3800
ms per ticket,0.03,3.1700,5.6100,77.8000


## 4. Where each one fails

The headline hides the interesting part. The shape to look for:

* **formats** (`EMAIL`, `IBAN`, `CREDIT_CARD`, `IP_ADDRESS`, `PHONE`) - rules
  and Presidio near 1.0, the NER checkpoint near 0, because it was never
  trained on them;
* **`PERSON`** - the reverse, and the only entity where a model earns its
  dependency;
* **local identifiers** (`EMPLOYEE_ID`, `NATIONAL_ID`, `DATE_OF_BIRTH`) - only
  the rule engine and GLiNER can know a *matricule* exists. Presidio has no
  recogniser for one and should sit at 0.

**That last row is why the detectors run alone.** Composed, the rule layer
underneath would cover it and the table would report Presidio finding the
matricule.

In [5]:
per_entity = pd.DataFrame(
    {
        backend: {
            entity: metrics['recall']
            for entity, metrics in run['detection']['per_entity'].items()
        }
        for backend, run in runs.items()
    }
).join(support).sort_values('annotated values', ascending=False)

columns = list(runs)
per_entity.style.format('{:.3f}', subset=columns).background_gradient(
    cmap='RdYlGn', vmin=0, vmax=1, subset=columns
)

,rules,transformer,presidio,gliner,annotated values
PERSON,1.000,0.988,0.625,0.988,168
PHONE,1.000,0.000,0.843,1.000,51
ADDRESS,1.000,0.000,0.000,1.000,28
NATIONAL_ID,1.000,0.000,0.000,1.000,21
DATE_OF_BIRTH,1.000,0.000,0.000,1.000,20
IBAN,1.000,0.000,0.000,0.579,19
IP_ADDRESS,1.000,0.000,1.000,0.000,19
CREDIT_CARD,1.000,0.000,0.000,1.000,17
EMPLOYEE_ID,1.000,0.000,0.000,0.706,17
EMAIL,1.000,0.000,0.000,1.000,13


## 5. The number that actually decides

**The table above is rigged in the rule engine's favour, and saying so is not
enough.** The generator samples names from `itsm/data/catalog.py` and the
gazetteer is built from the same lists, so its `PERSON` recall is near-perfect
*by construction*. A real corpus is full of names no list contains.

The probe re-runs every detector on twelve invented names spanning naming
traditions a French/English list will never cover, half in sentences with a
trigger word and half without. Small and synthetic: it proves a **direction**,
not a number.

In [6]:
from itsm.pipelines.eval_privacy import unseen_name_benchmark

probe = unseen_name_benchmark(settings, tuple(runs), isolate=True)

pd.DataFrame(
    {
        backend: {
            'recall on unseen names': round(entry['recall'], 4),
            'names never caught': len(entry['names_never_caught']),
        }
        for backend, entry in probe['backends'].items()
    }
)

/mnt/c/Users/AmineDIAB/projects/Thales/ITSM/src/itsm/privacy/gliner.py:281: FutureWarning: GLiNER.batch_predict_entities is deprecated and will be removed in a future release. Please use GLiNER.inference instead.
  raw = batched(chunk, names, threshold=self.min_score)


,rules,transformer,presidio,gliner
recall on unseen names,0.6667,0.9444,0.6389,1.0
names never caught,0.0000,0.0000,0.0000,0.0


## 6. Error analysis

What was missed, and what was destroyed that should not have been.
Over-redaction is the safer direction but not free: it eats the product
vocabulary the category classifier lives on (notebook 3).

In [7]:
for backend, run in runs.items():
    residual = run['residual']
    print('===', backend, '===')
    print('  still readable:', residual['residual_values'], 'of', residual['checked_values'])
    for item in residual['examples'][:4]:
        print('     leaked:', item)
    for value, count in run['detection']['top_false_positives'][:4]:
        print('     over-redacted:', count, 'x', value)
    print()

=== rules ===
  still readable: 0 of 373

=== transformer ===
  still readable: 207 of 373
     leaked: {'entity': 'PHONE', 'value': '+33 1 98 38 26 78', 'kind': 'full'}
     leaked: {'entity': 'PHONE', 'value': '+33 7 73 90 23 99', 'kind': 'full'}
     leaked: {'entity': 'ADDRESS', 'value': '17 boulevard Jean Jaures, 76091 Toulouse', 'kind': 'full'}
     leaked: {'entity': 'IP_ADDRESS', 'value': '10.59.207.59', 'kind': 'full'}
     over-redacted: 6 x PERSON: Cannot
     over-redacted: 4 x PERSON: Can
     over-redacted: 1 x PERSON: Com

=== presidio ===
  still readable: 164 of 373
     leaked: {'entity': 'CREDIT_CARD', 'value': '4816 3447 3243 2183', 'kind': 'full'}
     leaked: {'entity': 'NATIONAL_ID', 'value': '1 72 08 07 278 104 10', 'kind': 'full'}
     leaked: {'entity': 'EMPLOYEE_ID', 'value': 'EMP-24231', 'kind': 'full'}
     leaked: {'entity': 'IBAN', 'value': 'FR95 4893 3556 5502 2276 2400 398', 'kind': 'full'}
     over-redacted: 8 x PERSON: Numero de securite
     over-re

## 7. Conclusion

*Fill in from the run - the sentences below are the shape of the argument, not
its conclusion.*

**What ships.** The expected answer is not *pick one* but **formats from the
rule layer, names from whichever model wins section 5** - which is what the
composed `transformer` backend already does. Isolation is how the choice gets
made; composition is what runs.

**Read in this order:**

1. *Does anything survive?* One surviving name, mailbox, phone, NIR, IBAN or
   card is a breach; a surviving matricule is a quality issue. Never average
   the two.
2. *Where does it fail, and does that matter here?* 0 on `EMPLOYEE_ID` is fine
   if the extract has none, disqualifying if every third ticket quotes one.
3. *What does it cost?* Dependency, download, ms/ticket, over-redaction.

**Threats to validity:**

* `PERSON` recall is optimistic for `rules` by construction (section 5).
* Presidio runs **one language at a time** (`ITSM_PRESIDIO_LANGUAGE`) and this
  corpus is FR/EN. Run it under both to price that.
* GLiNER's entity names are a prompt; changing the wording changes the result.
* Generated data. A few hundred hand-annotated real tickets beat all of it.

In [8]:
settings.ensure_dirs()
path = settings.reports_dir / 'experiment_01_pii.json'
path.write_text(
    json.dumps(
        {
            'column': COLUMN,
            'n_tickets': len(sample),
            'isolated': True,
            'headline': comparison.to_dict(),
            'per_entity_recall': per_entity.to_dict(),
            'unseen_name_probe': probe['backends'],
        },
        indent=2,
        default=str,
    ),
    encoding='utf-8',
)
print('written to', path)

written to reports/experiment_01_pii.json
